# Zero-Shot Baseline Evaluation for Bengali Coreference Resolution

**Author:** Zenith Biswas
**Date:** December 2025  
**Purpose:** Establish zero-shot baselines on test set (71 documents) for comparison with fine-tuned models

---

## Experiment Configuration

| Model | Category | Optimal Threshold | HuggingFace ID |
|-------|----------|-------------------|----------------|
| mBERT | Multilingual | 0.575 | bert-base-multilingual-cased |
| BanglaBERT-Base | Bengali-specific | 0.860 | csebuetnlp/banglabert |
| RemBERT | Low-resource | 0.525 | google/rembert |
| MuRIL-Large | Indic-focused | 0.700 | google/muril-large-cased |
| BERT-Base-Uncased | Control | 0.800 | bert-base-uncased |

**Clustering:** Graph-based Connected Components  
**Embedding Strategy:** Span [start || end || mean]  
**Evaluation:** CoNLL-2012 metrics (MUC, B³, CEAFe, CoNLL F1)

In [1]:
"""
Cell 1: Setup and Environment Check
"""
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

# Core imports
import json
import re
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from collections import defaultdict
from datetime import datetime
from tqdm import tqdm
import time
import gc

# ML imports
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import networkx as nx

# Set random seeds
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Environment check
print("=" * 80)
print("ZERO-SHOT BASELINE EVALUATION")
print("=" * 80)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nEnvironment:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 80)

ZERO-SHOT BASELINE EVALUATION
Timestamp: 2025-12-14 23:38:00

Environment:
  PyTorch: 2.9.0+cu128
  CUDA available: True
  GPU: NVIDIA L4
  GPU Memory: 23.58 GB


In [2]:
"""
Cell 2: Experiment Configuration
"""

# =============================================================================
# PATHS - UPDATE THESE FOR YOUR ENVIRONMENT
# =============================================================================
BASE_DIR = Path('/teamspace/studios/this_studio/final_experiments')
TEST_FILE = BASE_DIR / 'test.conll'
OUTPUT_DIR = BASE_DIR / 'zero_shot_baseline'
OUTPUT_DIR.mkdir(exist_ok=True)

# =============================================================================
# MODEL CONFIGURATIONS (from zero-shot model selection experiment)
# =============================================================================
@dataclass
class ModelConfig:
    name: str
    hf_model_id: str
    category: str
    optimal_threshold: float
    hidden_dim: int

MODELS = [
    ModelConfig(
        name="mBERT",
        hf_model_id="google-bert/bert-base-multilingual-cased",
        category="multilingual",
        optimal_threshold=0.575,
        hidden_dim=768
    ),
    ModelConfig(
        name="BanglaBERT-Base",
        hf_model_id="csebuetnlp/banglabert",
        category="bengali-specific",
        optimal_threshold=0.860,
        hidden_dim=768
    ),
    ModelConfig(
        name="RemBERT",
        hf_model_id="google/rembert",
        category="low-resource",
        optimal_threshold=0.525,
        hidden_dim=1152
    ),
    ModelConfig(
        name="MuRIL-Large",
        hf_model_id="google/muril-large-cased",
        category="indic-focused",
        optimal_threshold=0.700,
        hidden_dim=1024
    ),
    ModelConfig(
        name="BERT-Base-Uncased",
        hf_model_id="google-bert/bert-base-uncased",
        category="control",
        optimal_threshold=0.800,
        hidden_dim=768
    ),
]

print("=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)
print(f"\nTest file: {TEST_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"\nModels to evaluate ({len(MODELS)}):")
print("-" * 80)
print(f"{'Model':<20} {'Category':<18} {'Threshold':>10} {'Hidden Dim':>12}")
print("-" * 80)
for m in MODELS:
    print(f"{m.name:<20} {m.category:<18} {m.optimal_threshold:>10.3f} {m.hidden_dim:>12}")
print("=" * 80)

EXPERIMENT CONFIGURATION

Test file: /teamspace/studios/this_studio/final_experiments/test.conll
Output dir: /teamspace/studios/this_studio/final_experiments/zero_shot_baseline

Models to evaluate (5):
--------------------------------------------------------------------------------
Model                Category            Threshold   Hidden Dim
--------------------------------------------------------------------------------
mBERT                multilingual            0.575          768
BanglaBERT-Base      bengali-specific        0.860          768
RemBERT              low-resource            0.525         1152
MuRIL-Large          indic-focused           0.700         1024
BERT-Base-Uncased    control                 0.800          768


In [3]:
"""
Cell 3: CoNLL Parser for Coreference
"""

def parse_conll_file(filepath: Path) -> List[Dict]:
    """
    Parse CoNLL-2012 format file and extract documents with mentions and clusters.
    
    CoNLL format columns:
    0: Document ID
    1: Part number
    2: Word number
    3: Word/Token
    ...
    -1: Coreference column (e.g., (1), (1), (1|2), etc.)
    
    Returns:
        List of document dicts with keys:
        - 'id': document ID
        - 'sentences': list of token lists
        - 'mentions': list of {sentence_idx, start, end}
        - 'clusters': list of mention index lists (gold clusters)
    """
    documents = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split by document
    doc_pattern = r'#begin document \((.+?)\).*?\n(.*?)#end document'
    doc_matches = re.findall(doc_pattern, content, re.DOTALL)
    
    for doc_id, doc_content in doc_matches:
        sentences = []
        current_sentence = []
        
        # Track mentions: {cluster_id: [(sent_idx, start, end), ...]}
        cluster_mentions = defaultdict(list)
        # Stack for nested mentions: {cluster_id: start_position}
        open_mentions = defaultdict(list)
        
        word_idx_in_sent = 0
        sent_idx = 0
        
        for line in doc_content.strip().split('\n'):
            if not line.strip():
                # Sentence boundary
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                    sent_idx += 1
                    word_idx_in_sent = 0
                continue
            
            parts = line.split()
            if len(parts) < 4:
                continue
            
            token = parts[3]
            coref_col = parts[-1]  # Last column is coreference
            
            current_sentence.append(token)
            
            # Parse coreference column
            if coref_col != '-' and coref_col != '_':
                # Handle multiple annotations separated by |
                annotations = coref_col.split('|')
                for ann in annotations:
                    # Opening: (123 or (123)
                    # Closing: 123) or (123)
                    
                    # Check for (N) - single token mention
                    single_match = re.match(r'^\((\d+)\)$', ann)
                    if single_match:
                        cluster_id = int(single_match.group(1))
                        cluster_mentions[cluster_id].append(
                            (sent_idx, word_idx_in_sent, word_idx_in_sent)
                        )
                        continue
                    
                    # Check for (N - opening
                    open_match = re.match(r'^\((\d+)$', ann)
                    if open_match:
                        cluster_id = int(open_match.group(1))
                        open_mentions[cluster_id].append((sent_idx, word_idx_in_sent))
                        continue
                    
                    # Check for N) - closing
                    close_match = re.match(r'^(\d+)\)$', ann)
                    if close_match:
                        cluster_id = int(close_match.group(1))
                        if open_mentions[cluster_id]:
                            start_sent, start_word = open_mentions[cluster_id].pop()
                            if start_sent == sent_idx:  # Same sentence
                                cluster_mentions[cluster_id].append(
                                    (sent_idx, start_word, word_idx_in_sent)
                                )
            
            word_idx_in_sent += 1
        
        # Don't forget last sentence
        if current_sentence:
            sentences.append(current_sentence)
        
        # Convert to mention list and cluster indices
        all_mentions = []
        mention_to_idx = {}
        clusters = []
        
        for cluster_id, mention_spans in cluster_mentions.items():
            cluster_indices = []
            for sent_idx, start, end in mention_spans:
                key = (sent_idx, start, end)
                if key not in mention_to_idx:
                    mention_to_idx[key] = len(all_mentions)
                    all_mentions.append({
                        'sentence_idx': sent_idx,
                        'start': start,
                        'end': end  # inclusive
                    })
                cluster_indices.append(mention_to_idx[key])
            if cluster_indices:
                clusters.append(cluster_indices)
        
        documents.append({
            'id': doc_id,
            'sentences': sentences,
            'mentions': all_mentions,
            'clusters': clusters
        })
    
    return documents

# Load test data
print("Loading test data...")
test_docs = parse_conll_file(TEST_FILE)

# Statistics
total_mentions = sum(len(doc['mentions']) for doc in test_docs)
total_clusters = sum(len(doc['clusters']) for doc in test_docs)
avg_clusters = total_clusters / len(test_docs) if test_docs else 0

print("\n" + "=" * 80)
print("TEST SET STATISTICS")
print("=" * 80)
print(f"Documents: {len(test_docs)}")
print(f"Total mentions: {total_mentions:,}")
print(f"Total clusters: {total_clusters}")
print(f"Avg clusters/doc: {avg_clusters:.2f}")

# Sample verification
if test_docs:
    doc = test_docs[0]
    print(f"\nSample document: {doc['id']}")
    print(f"  Sentences: {len(doc['sentences'])}")
    print(f"  Mentions: {len(doc['mentions'])}")
    print(f"  Clusters: {len(doc['clusters'])}")
    if doc['mentions']:
        m = doc['mentions'][0]
        tokens = doc['sentences'][m['sentence_idx']][m['start']:m['end']+1]
        print(f"  First mention: '{' '.join(tokens)}'")
print("=" * 80)

Loading test data...

TEST SET STATISTICS
Documents: 71
Total mentions: 3,308
Total clusters: 282
Avg clusters/doc: 3.97

Sample document: train_010
  Sentences: 14
  Mentions: 30
  Clusters: 4
  First mention: 'গোপাল ভাঁড়'


In [4]:
"""
Cell 4: Coreference Evaluation Metrics (CoNLL-2012)
"""

class CorefMetrics:
    """
    Standard coreference evaluation metrics:
    - MUC (Vilain et al., 1995): Link-based
    - B³ (Bagga & Baldwin, 1998): Mention-based
    - CEAFe (Luo, 2005): Entity-based with optimal alignment
    - CoNLL F1: Average of MUC, B³, CEAFe
    """
    
    @staticmethod
    def muc(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        """MUC metric."""
        def links(clusters, reference_clusters):
            total_mentions = sum(len(c) for c in clusters)
            total_partitions = 0
            for cluster in clusters:
                cluster_set = set(cluster)
                partitions = sum(1 for ref in reference_clusters if cluster_set & set(ref))
                total_partitions += partitions
            return total_mentions - total_partitions
        
        gold_mentions = sum(len(c) for c in gold_clusters)
        gold_links = gold_mentions - len(gold_clusters)
        
        recall = links(gold_clusters, pred_clusters) / gold_links if gold_links > 0 else 0.0
        
        pred_mentions = sum(len(c) for c in pred_clusters)
        pred_links = pred_mentions - len(pred_clusters)
        
        precision = links(pred_clusters, gold_clusters) / pred_links if pred_links > 0 else 0.0
        
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def b_cubed(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        """B³ metric."""
        mention_to_gold = {}
        for cluster in gold_clusters:
            cluster_set = frozenset(cluster)
            for mention in cluster:
                mention_to_gold[mention] = cluster_set
        
        mention_to_pred = {}
        for cluster in pred_clusters:
            cluster_set = frozenset(cluster)
            for mention in cluster:
                mention_to_pred[mention] = cluster_set
        
        all_mentions = set(mention_to_gold.keys()) | set(mention_to_pred.keys())
        
        if not all_mentions:
            return 0.0, 0.0, 0.0
        
        total_precision = 0.0
        total_recall = 0.0
        
        for mention in all_mentions:
            gold_cluster = mention_to_gold.get(mention, frozenset([mention]))
            pred_cluster = mention_to_pred.get(mention, frozenset([mention]))
            intersection = len(gold_cluster & pred_cluster)
            
            if len(pred_cluster) > 0:
                total_precision += intersection / len(pred_cluster)
            if len(gold_cluster) > 0:
                total_recall += intersection / len(gold_cluster)
        
        precision = total_precision / len(all_mentions)
        recall = total_recall / len(all_mentions)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def ceaf_e(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Tuple[float, float, float]:
        """CEAFe metric with optimal entity alignment."""
        if not gold_clusters or not pred_clusters:
            return 0.0, 0.0, 0.0
        
        n_gold, n_pred = len(gold_clusters), len(pred_clusters)
        similarity_matrix = np.zeros((n_gold, n_pred))
        
        for i, gold_cluster in enumerate(gold_clusters):
            gold_set = set(gold_cluster)
            for j, pred_cluster in enumerate(pred_clusters):
                pred_set = set(pred_cluster)
                intersection = len(gold_set & pred_set)
                if len(gold_set) + len(pred_set) > 0:
                    similarity_matrix[i, j] = 2 * intersection / (len(gold_set) + len(pred_set))
        
        row_ind, col_ind = linear_sum_assignment(-similarity_matrix)
        total_similarity = similarity_matrix[row_ind, col_ind].sum()
        
        recall = total_similarity / n_gold if n_gold > 0 else 0.0
        precision = total_similarity / n_pred if n_pred > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        return precision, recall, f1
    
    @staticmethod
    def evaluate(gold_clusters: List[List[int]], pred_clusters: List[List[int]]) -> Dict[str, float]:
        """Compute all metrics."""
        # Filter singletons for MUC
        gold_non_singleton = [c for c in gold_clusters if len(c) > 1]
        pred_non_singleton = [c for c in pred_clusters if len(c) > 1]
        
        muc_p, muc_r, muc_f1 = CorefMetrics.muc(gold_non_singleton, pred_non_singleton)
        b3_p, b3_r, b3_f1 = CorefMetrics.b_cubed(gold_clusters, pred_clusters)
        ceaf_p, ceaf_r, ceaf_f1 = CorefMetrics.ceaf_e(gold_clusters, pred_clusters)
        
        conll_f1 = (muc_f1 + b3_f1 + ceaf_f1) / 3
        
        return {
            'MUC_P': muc_p * 100, 'MUC_R': muc_r * 100, 'MUC_F1': muc_f1 * 100,
            'B3_P': b3_p * 100, 'B3_R': b3_r * 100, 'B3_F1': b3_f1 * 100,
            'CEAF_P': ceaf_p * 100, 'CEAF_R': ceaf_r * 100, 'CEAF_F1': ceaf_f1 * 100,
            'CoNLL_F1': conll_f1 * 100
        }

# Verify metrics
print("Verifying metrics implementation...")
gold_test = [[0, 1, 2], [3, 4]]
pred_perfect = [[0, 1, 2], [3, 4]]
m = CorefMetrics.evaluate(gold_test, pred_perfect)
print(f"  Perfect match: CoNLL F1 = {m['CoNLL_F1']:.1f}% (expected: 100%)")

if test_docs:
    doc = test_docs[0]
    m = CorefMetrics.evaluate(doc['clusters'], doc['clusters'])
    print(f"  Self-evaluation on '{doc['id']}': CoNLL F1 = {m['CoNLL_F1']:.1f}%")

print("✓ Metrics verified")

Verifying metrics implementation...
  Perfect match: CoNLL F1 = 100.0% (expected: 100%)
  Self-evaluation on 'train_010': CoNLL F1 = 100.0%
✓ Metrics verified


In [5]:
"""
Cell 5: Graph-based Connected Components Clustering
"""

def graph_connected_components(similarity_matrix: np.ndarray, threshold: float) -> List[List[int]]:
    """
    Graph-based Connected Components clustering.
    
    - Build graph where nodes = mentions
    - Add edge if similarity >= threshold
    - Clusters = connected components (transitive closure)
    
    Key property: If A~B and B~C, then {A,B,C} form one cluster even if A≁C.
    This naturally captures coreference chains.
    """
    n_mentions = similarity_matrix.shape[0]
    if n_mentions == 0:
        return []
    if n_mentions == 1:
        return [[0]]
    
    # Build graph
    G = nx.Graph()
    G.add_nodes_from(range(n_mentions))
    
    # Add edges where similarity >= threshold
    for i in range(n_mentions):
        for j in range(i + 1, n_mentions):
            if similarity_matrix[i, j] >= threshold:
                G.add_edge(i, j)
    
    # Find connected components
    clusters = [list(component) for component in nx.connected_components(G)]
    
    return clusters

# Test clustering
test_sim = np.array([
    [1.0, 0.9, 0.3, 0.2],
    [0.9, 1.0, 0.3, 0.2],
    [0.3, 0.3, 1.0, 0.8],
    [0.2, 0.2, 0.8, 1.0],
])
clusters = graph_connected_components(test_sim, 0.5)
print(f"Test clustering (threshold=0.5): {sorted([sorted(c) for c in clusters])}")
print("Expected: [[0, 1], [2, 3]]")
print("✓ Clustering verified")

Test clustering (threshold=0.5): [[0, 1], [2, 3]]
Expected: [[0, 1], [2, 3]]
✓ Clustering verified


In [6]:
"""
Cell 6: Embedding Extractor with Span Strategy
"""

class EmbeddingExtractor:
    """
    Extract span embeddings [start || end || mean] for mentions.
    """
    
    def __init__(self, model, tokenizer, device='cuda'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model.eval()
    
    def get_mention_embeddings(self, doc: Dict) -> np.ndarray:
        """
        Extract span embeddings for all mentions in a document.
        Span embedding = [start_token || end_token || mean_tokens]
        """
        if not doc['mentions']:
            return np.array([]).reshape(0, 0)
        
        embeddings = []
        
        for mention in doc['mentions']:
            sent_idx = mention['sentence_idx']
            start = mention['start']
            end = mention['end']  # inclusive
            
            sentence = doc['sentences'][sent_idx]
            
            # Tokenize sentence
            encoding = self.tokenizer(
                sentence,
                is_split_into_words=True,
                return_tensors='pt',
                truncation=True,
                max_length=512,
                padding=True
            )
            encoding = {k: v.to(self.device) for k, v in encoding.items()}
            
            # Get word_ids mapping
            word_ids = self.tokenizer(
                sentence,
                is_split_into_words=True,
                truncation=True,
                max_length=512
            ).word_ids()
            
            # Find token positions for mention span
            positions = []
            for tok_idx, word_idx in enumerate(word_ids):
                if word_idx is not None and start <= word_idx <= end:
                    positions.append(tok_idx)
            
            # Fallback if no positions found
            if not positions:
                positions = [1]  # Skip [CLS]
            
            # Get hidden states
            with torch.no_grad():
                outputs = self.model(**encoding)
                hidden_states = outputs.last_hidden_state[0]  # (seq_len, hidden_dim)
            
            # Span embedding: [start, end, mean]
            start_emb = hidden_states[positions[0]]
            end_emb = hidden_states[positions[-1]]
            mean_emb = hidden_states[positions].mean(dim=0)
            mention_emb = torch.cat([start_emb, end_emb, mean_emb])
            
            embeddings.append(mention_emb.cpu().numpy())
        
        return np.array(embeddings)
    
    def compute_similarity_matrix(self, embeddings: np.ndarray) -> np.ndarray:
        """Compute pairwise cosine similarity."""
        if len(embeddings) == 0:
            return np.array([]).reshape(0, 0)
        return cosine_similarity(embeddings)

print("✓ EmbeddingExtractor defined")

✓ EmbeddingExtractor defined


In [7]:
"""
Cell 7: Zero-Shot Evaluator
"""

def evaluate_model_zero_shot(
    model_config: ModelConfig,
    documents: List[Dict],
    device: str = 'cuda'
) -> Dict:
    """
    Evaluate a model in zero-shot setting on given documents.
    
    Returns dict with:
    - aggregate metrics
    - per-document results
    - timing info
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_config.name}")
    print(f"{'='*60}")
    print(f"  HuggingFace ID: {model_config.hf_model_id}")
    print(f"  Category: {model_config.category}")
    print(f"  Threshold: {model_config.optimal_threshold}")
    
    start_time = time.time()
    
    # Load model
    print(f"\n  Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_config.hf_model_id, trust_remote_code=True)
    model = model.to(device)
    model.eval()
    
    extractor = EmbeddingExtractor(model, tokenizer, device)
    
    load_time = time.time() - start_time
    print(f"  Model loaded in {load_time:.1f}s")
    
    # Evaluate each document
    per_doc_results = []
    all_gold_clusters = []
    all_pred_clusters = []
    
    print(f"\n  Processing {len(documents)} documents...")
    
    for doc in tqdm(documents, desc=f"  {model_config.name}"):
        if not doc['mentions'] or not doc['clusters']:
            continue
        
        # Get embeddings
        embeddings = extractor.get_mention_embeddings(doc)
        
        if embeddings.size == 0:
            continue
        
        # Compute similarity and cluster
        sim_matrix = extractor.compute_similarity_matrix(embeddings)
        pred_clusters = graph_connected_components(sim_matrix, model_config.optimal_threshold)
        
        # Evaluate
        gold_clusters = doc['clusters']
        metrics = CorefMetrics.evaluate(gold_clusters, pred_clusters)
        
        per_doc_results.append({
            'doc_id': doc['id'],
            'n_mentions': len(doc['mentions']),
            'n_gold_clusters': len(gold_clusters),
            'n_pred_clusters': len(pred_clusters),
            **metrics
        })
        
        # Track for aggregate computation
        # Offset mention indices for aggregate computation
        offset = sum(len(c) for c in all_gold_clusters) if all_gold_clusters else 0
        all_gold_clusters.extend([[m + offset for m in c] for c in gold_clusters])
        all_pred_clusters.extend([[m + offset for m in c] for c in pred_clusters])
    
    total_time = time.time() - start_time
    
    # Compute aggregate metrics
    aggregate_metrics = CorefMetrics.evaluate(all_gold_clusters, all_pred_clusters)
    
    # Also compute micro-average from per-doc results
    df = pd.DataFrame(per_doc_results)
    macro_avg = {
        'macro_MUC_F1': df['MUC_F1'].mean(),
        'macro_B3_F1': df['B3_F1'].mean(),
        'macro_CEAF_F1': df['CEAF_F1'].mean(),
        'macro_CoNLL_F1': df['CoNLL_F1'].mean(),
    }
    
    avg_clusters_per_doc = df['n_pred_clusters'].mean()
    
    # Print summary
    print(f"\n  Results:")
    print(f"  {'-'*50}")
    print(f"  CoNLL F1 (aggregate): {aggregate_metrics['CoNLL_F1']:.2f}%")
    print(f"  CoNLL F1 (macro avg): {macro_avg['macro_CoNLL_F1']:.2f}%")
    print(f"  MUC F1:  {aggregate_metrics['MUC_F1']:.2f}%")
    print(f"  B³ F1:   {aggregate_metrics['B3_F1']:.2f}%")
    print(f"  CEAF F1: {aggregate_metrics['CEAF_F1']:.2f}%")
    print(f"  Avg clusters/doc: {avg_clusters_per_doc:.2f} (gold: {df['n_gold_clusters'].mean():.2f})")
    print(f"  Total time: {total_time:.1f}s")
    
    # Clear GPU memory
    del model, tokenizer, extractor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return {
        'model_name': model_config.name,
        'model_id': model_config.hf_model_id,
        'category': model_config.category,
        'threshold': model_config.optimal_threshold,
        'n_documents': len(per_doc_results),
        'aggregate_metrics': aggregate_metrics,
        'macro_metrics': macro_avg,
        'avg_clusters_per_doc': avg_clusters_per_doc,
        'per_document_results': per_doc_results,
        'total_time_seconds': total_time,
    }

print("✓ Evaluator function defined")

✓ Evaluator function defined


In [8]:
"""
Cell 8: Run Zero-Shot Evaluation on All 5 Models
"""

print("=" * 80)
print("RUNNING ZERO-SHOT BASELINE EVALUATION")
print("=" * 80)
print(f"\nTest documents: {len(test_docs)}")
print(f"Models to evaluate: {len(MODELS)}")
print(f"Output directory: {OUTPUT_DIR}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

all_results = []
experiment_start = time.time()

for i, model_config in enumerate(MODELS):
    print(f"\n[{i+1}/{len(MODELS)}] Processing {model_config.name}...")
    
    result = evaluate_model_zero_shot(model_config, test_docs, device)
    all_results.append(result)

experiment_time = time.time() - experiment_start
print(f"\n{'='*80}")
print(f"EXPERIMENT COMPLETE")
print(f"Total time: {experiment_time/60:.1f} minutes")
print(f"{'='*80}")

RUNNING ZERO-SHOT BASELINE EVALUATION

Test documents: 71
Models to evaluate: 5
Output directory: /teamspace/studios/this_studio/final_experiments/zero_shot_baseline
Device: cuda

[1/5] Processing mBERT...

Evaluating: mBERT
  HuggingFace ID: google-bert/bert-base-multilingual-cased
  Category: multilingual
  Threshold: 0.575

  Loading model...
  Model loaded in 2.0s

  Processing 71 documents...


  mBERT: 100%|██████████| 71/71 [00:42<00:00,  1.69it/s]



  Results:
  --------------------------------------------------
  CoNLL F1 (aggregate): 63.61%
  CoNLL F1 (macro avg): 64.17%
  MUC F1:  95.85%
  B³ F1:   63.32%
  CEAF F1: 31.67%
  Avg clusters/doc: 3.27 (gold: 3.97)
  Total time: 44.1s

[2/5] Processing BanglaBERT-Base...

Evaluating: BanglaBERT-Base
  HuggingFace ID: csebuetnlp/banglabert
  Category: bengali-specific
  Threshold: 0.86

  Loading model...
  Model loaded in 2.0s

  Processing 71 documents...


  BanglaBERT-Base: 100%|██████████| 71/71 [00:50<00:00,  1.41it/s]



  Results:
  --------------------------------------------------
  CoNLL F1 (aggregate): 62.38%
  CoNLL F1 (macro avg): 63.62%
  MUC F1:  95.63%
  B³ F1:   62.90%
  CEAF F1: 28.61%
  Avg clusters/doc: 3.73 (gold: 3.97)
  Total time: 52.2s

[3/5] Processing RemBERT...

Evaluating: RemBERT
  HuggingFace ID: google/rembert
  Category: low-resource
  Threshold: 0.525

  Loading model...
  Model loaded in 7.3s

  Processing 71 documents...


...

  Model loaded in 5.9s

  Processing 71 documents...


  MuRIL-Large: 100%|██████████| 71/71 [01:13<00:00,  1.03s/it]



  Results:
  --------------------------------------------------
  CoNLL F1 (aggregate): 62.44%
  CoNLL F1 (macro avg): 64.08%
  MUC F1:  94.76%
  B³ F1:   63.10%
  CEAF F1: 29.45%
  Avg clusters/doc: 4.80 (gold: 3.97)
  Total time: 79.1s

[5/5] Processing BERT-Base-Uncased...

Evaluating: BERT-Base-Uncased
  HuggingFace ID: google-bert/bert-base-uncased
  Category: control
  Threshold: 0.8

  Loading model...
  Model loaded in 0.9s

  Processing 71 documents...


  BERT-Base-Uncased: 100%|██████████| 71/71 [00:42<00:00,  1.67it/s]



  Results:
  --------------------------------------------------
  CoNLL F1 (aggregate): 63.82%
  CoNLL F1 (macro avg): 64.06%
  MUC F1:  94.86%
  B³ F1:   64.37%
  CEAF F1: 32.22%
  Avg clusters/doc: 3.94 (gold: 3.97)
  Total time: 43.4s

EXPERIMENT COMPLETE
Total time: 5.8 minutes


In [9]:
"""
Cell 9: Summary and Comparison Table
"""

print("\n" + "=" * 80)
print("ZERO-SHOT BASELINE RESULTS SUMMARY")
print("=" * 80)

# Create summary table
summary_data = []
for result in all_results:
    summary_data.append({
        'Model': result['model_name'],
        'Category': result['category'],
        'Threshold': result['threshold'],
        'CoNLL F1': result['aggregate_metrics']['CoNLL_F1'],
        'MUC F1': result['aggregate_metrics']['MUC_F1'],
        'B³ F1': result['aggregate_metrics']['B3_F1'],
        'CEAF F1': result['aggregate_metrics']['CEAF_F1'],
        'Avg Clusters': result['avg_clusters_per_doc'],
    })

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values('CoNLL F1', ascending=False).reset_index(drop=True)
df_summary.index = df_summary.index + 1  # 1-based ranking

print("\nRanking by CoNLL F1 Score:")
print("-" * 100)
print(df_summary.to_string())
print("-" * 100)

# Gold standard clusters per doc
gold_clusters_avg = sum(len(doc['clusters']) for doc in test_docs) / len(test_docs)
print(f"\nGold standard: {gold_clusters_avg:.2f} clusters/document")

# Best model
best_model = df_summary.iloc[0]['Model']
best_score = df_summary.iloc[0]['CoNLL F1']
print(f"\nBest performing model: {best_model} ({best_score:.2f}% CoNLL F1)")


ZERO-SHOT BASELINE RESULTS SUMMARY

Ranking by CoNLL F1 Score:
----------------------------------------------------------------------------------------------------
               Model          Category  Threshold   CoNLL F1     MUC F1      B³ F1    CEAF F1  Avg Clusters
1  BERT-Base-Uncased           control      0.800  63.817001  94.858542  64.367597  32.224863      3.943662
2              mBERT      multilingual      0.575  63.614743  95.849732  63.322007  31.672489      3.267606
3            RemBERT      low-resource      0.525  63.201741  94.915321  62.594986  32.094915      3.422535
4        MuRIL-Large     indic-focused      0.700  62.437312  94.755296  63.103111  29.453529      4.802817
5    BanglaBERT-Base  bengali-specific      0.860  62.378061  95.628706  62.896250  28.609226      3.732394
----------------------------------------------------------------------------------------------------

Gold standard: 3.97 clusters/document

Best performing model: BERT-Base-Uncased (63.8

In [10]:
"""
Cell 10: Save Results
"""

# Save summary CSV
summary_file = OUTPUT_DIR / 'zero_shot_baseline_summary.csv'
df_summary.to_csv(summary_file, index=True)
print(f"Saved: {summary_file}")

# Save complete results JSON
complete_results = {
    'experiment_name': 'Zero-Shot Baseline Evaluation',
    'date': datetime.now().isoformat(),
    'test_set': {
        'file': str(TEST_FILE),
        'n_documents': len(test_docs),
        'n_mentions': sum(len(doc['mentions']) for doc in test_docs),
        'n_clusters': sum(len(doc['clusters']) for doc in test_docs),
    },
    'configuration': {
        'clustering': 'graph_connected_components',
        'embedding_strategy': 'span [start || end || mean]',
    },
    'results': [
        {
            'model_name': r['model_name'],
            'category': r['category'],
            'threshold': r['threshold'],
            'metrics': r['aggregate_metrics'],
            'per_document': r['per_document_results'],  # Kept here if needed later
        }
        for r in all_results
    ],
}

results_file = OUTPUT_DIR / 'zero_shot_baseline_results.json'
with open(results_file, 'w') as f:
    json.dump(complete_results, f, indent=2, default=float)
print(f"Saved: {results_file}")

print(f"\n✓ Results saved to {OUTPUT_DIR}")

Saved: /teamspace/studios/this_studio/final_experiments/zero_shot_baseline/zero_shot_baseline_summary.csv
Saved: /teamspace/studios/this_studio/final_experiments/zero_shot_baseline/zero_shot_baseline_results.json

✓ Results saved to /teamspace/studios/this_studio/final_experiments/zero_shot_baseline


In [12]:
"""
Cell 11: Display Results for Thesis (with Domain-wise Breakdown)
"""

from IPython.display import display, Markdown

print("\n" + "=" * 80)
print("ZERO-SHOT BASELINE RESULTS")
print("=" * 80)

# =============================================================================
# OVERALL RESULTS TABLE
# =============================================================================
df_display = df_summary.copy()
df_display['CoNLL F1'] = df_display['CoNLL F1'].apply(lambda x: f"{x:.2f}%")
df_display['MUC F1'] = df_display['MUC F1'].apply(lambda x: f"{x:.2f}%")
df_display['B³ F1'] = df_display['B³ F1'].apply(lambda x: f"{x:.2f}%")
df_display['CEAF F1'] = df_display['CEAF F1'].apply(lambda x: f"{x:.2f}%")
df_display['Threshold'] = df_display['Threshold'].apply(lambda x: f"{x:.3f}")
df_display['Avg Clusters'] = df_display['Avg Clusters'].apply(lambda x: f"{x:.2f}")
df_display.index.name = 'Rank'

print("\n### Overall Performance on Test Set (n=71)\n")
display(df_display[['Model', 'Category', 'Threshold', 'CoNLL F1', 'MUC F1', 'B³ F1', 'CEAF F1', 'Avg Clusters']])

# =============================================================================
# DOMAIN-WISE RESULTS
# =============================================================================
print("\n" + "=" * 80)
print("DOMAIN-WISE BREAKDOWN")
print("=" * 80)

# Build domain mapping from split info
split_info_file = BASE_DIR / 'dataset_split_info.json'
with open(split_info_file, 'r') as f:
    split_info = json.load(f)

# Create doc_id -> domain mapping from test_ids in each domain
doc_to_domain = {}
for domain in ['biography', 'descriptive', 'novel', 'story']:
    for doc_id in split_info['bencoref_stratification'][domain]['test_ids']:
        doc_to_domain[doc_id] = domain

# Count documents per domain
domain_counts = defaultdict(int)
for doc in test_docs:
    doc['domain'] = doc_to_domain.get(doc['id'], 'unknown')
    domain_counts[doc['domain']] += 1

print("\nTest documents per domain:")
for domain in ['biography', 'descriptive', 'novel', 'story']:
    print(f"  {domain}: {domain_counts[domain]}")

# Compute domain-wise metrics for each model
domain_results = []

for result in all_results:
    model_name = result['model_name']
    per_doc = pd.DataFrame(result['per_document_results'])
    
    # Add domain info
    per_doc['domain'] = per_doc['doc_id'].apply(lambda x: doc_to_domain.get(x, 'unknown'))
    
    # Group by domain
    for domain in ['biography', 'descriptive', 'novel', 'story']:
        domain_df = per_doc[per_doc['domain'] == domain]
        if len(domain_df) > 0:
            domain_results.append({
                'Model': model_name,
                'Domain': domain,
                'N': len(domain_df),
                'CoNLL F1': domain_df['CoNLL_F1'].mean(),
                'MUC F1': domain_df['MUC_F1'].mean(),
                'B³ F1': domain_df['B3_F1'].mean(),
                'CEAF F1': domain_df['CEAF_F1'].mean(),
            })

df_domain = pd.DataFrame(domain_results)

# =============================================================================
# PIVOT TABLE: Models vs Domains (CoNLL F1)
# =============================================================================
print("\n### CoNLL F1 by Model × Domain\n")

pivot = df_domain.pivot(index='Model', columns='Domain', values='CoNLL F1')
# Reorder columns
pivot = pivot[['biography', 'descriptive', 'novel', 'story']]
# Reorder models by overall performance
model_order = df_summary['Model'].values
pivot = pivot.reindex(model_order)

# Format as percentages
pivot_display = pivot.map(lambda x: f"{x:.2f}%" if pd.notna(x) else "-")
display(pivot_display)

# =============================================================================
# DETAILED DOMAIN RESULTS
# =============================================================================
print("\n### Detailed Metrics by Domain\n")

for domain in ['biography', 'descriptive', 'novel', 'story']:
    domain_df = df_domain[df_domain['Domain'] == domain].copy()
    domain_df = domain_df.set_index('Model')
    domain_df = domain_df.reindex(model_order)
    
    # Format percentages
    for col in ['CoNLL F1', 'MUC F1', 'B³ F1', 'CEAF F1']:
        domain_df[col] = domain_df[col].apply(lambda x: f"{x:.2f}%")
    
    print(f"\n**{domain.upper()}** (n={domain_counts[domain]}):")
    display(domain_df[['CoNLL F1', 'MUC F1', 'B³ F1', 'CEAF F1']])

# =============================================================================
# NEXT STEPS
# =============================================================================
print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
Compare these zero-shot baselines against fine-tuned models trained on:
  1. transmucores_bencoref_train.conll
  2. transmucores_bencoref_full_bt_train.conll (+ back-translation)
  3. transmucores_bencoref_full_pp.conll (+ paraphrase)
""")
print("=" * 80)


ZERO-SHOT BASELINE RESULTS

### Overall Performance on Test Set (n=71)



,Model,Category,Threshold,CoNLL F1,MUC F1,B³ F1,CEAF F1,Avg Clusters
Rank,,,,,,,,
1,BERT-Base-Uncased,control,0.800,63.82%,94.86%,64.37%,32.22%,3.94
2,mBERT,multilingual,0.575,63.61%,95.85%,63.32%,31.67%,3.27
3,RemBERT,low-resource,0.525,63.20%,94.92%,62.59%,32.09%,3.42
4,MuRIL-Large,indic-focused,0.700,62.44%,94.76%,63.10%,29.45%,4.80
5,BanglaBERT-Base,bengali-specific,0.860,62.38%,95.63%,62.90%,28.61%,3.73



DOMAIN-WISE BREAKDOWN

Test documents per domain:
  biography: 10
  descriptive: 21
  novel: 8
  story: 32

### CoNLL F1 by Model × Domain



Domain,biography,descriptive,novel,story
Model,,,,
BERT-Base-Uncased,68.16%,65.86%,57.09%,63.34%
mBERT,69.69%,68.59%,58.00%,61.09%
RemBERT,64.76%,67.31%,58.75%,60.95%
MuRIL-Large,70.44%,68.64%,57.66%,60.69%
BanglaBERT-Base,72.99%,64.54%,57.97%,61.50%



### Detailed Metrics by Domain


**BIOGRAPHY** (n=10):


...


**NOVEL** (n=8):


,CoNLL F1,MUC F1,B³ F1,CEAF F1
Model,,,,
BERT-Base-Uncased,57.09%,90.92%,54.14%,26.22%
mBERT,58.00%,92.63%,52.41%,28.97%
RemBERT,58.75%,89.23%,51.95%,35.07%
MuRIL-Large,57.66%,88.35%,53.70%,30.94%
BanglaBERT-Base,57.97%,93.05%,54.10%,26.76%



**STORY** (n=32):


,CoNLL F1,MUC F1,B³ F1,CEAF F1
Model,,,,
BERT-Base-Uncased,63.34%,95.16%,61.38%,33.47%
mBERT,61.09%,95.71%,58.44%,29.13%
RemBERT,60.95%,95.47%,57.85%,29.52%
MuRIL-Large,60.69%,93.36%,59.59%,29.13%
BanglaBERT-Base,61.50%,95.39%,59.46%,29.65%



NEXT STEPS

Compare these zero-shot baselines against fine-tuned models trained on:
  1. transmucores_bencoref_train.conll
  2. transmucores_bencoref_full_bt_train.conll (+ back-translation)
  3. transmucores_bencoref_full_pp.conll (+ paraphrase)

